In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
import pandas as pd

master_train = pd.read_parquet('../data/master_train.parquet')
master_test = pd.read_parquet('../data/master_test.parquet')

print("Data loaded! Shape:", master_train.shape)

Data loaded! Shape: (307511, 318)


In [3]:
master_train

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CNT_INSTALMENT_FUTURE_std,CNT_INSTALMENT_FUTURE_mean,SK_DPD_min_y,SK_DPD_max_y,SK_DPD_std_y,SK_DPD_mean_y,SK_DPD_DEF_min_y,SK_DPD_DEF_max_y,SK_DPD_DEF_std_y,SK_DPD_DEF_mean_y
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,5.627314,15.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,3.842811,5.785714,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,1.707825,2.250000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,10.163272,8.650000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,6.312307,8.969697,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,2.669270,4.375000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,2.160247,3.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,1.903943,2.000000,0.0,5.0,1.212678,0.294118,0.0,5.0,1.212678,0.294118
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,3.360373,10.350000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000


In [4]:
from sklearn.model_selection import train_test_split

x = master_train.drop('TARGET',axis = 1)
y = master_train["TARGET"]

x_train , x_val , y_train , y_val = train_test_split(x,y,test_size= 0.2 , random_state = 42)

In [5]:
x_train = x_train.drop("SK_ID_CURR",axis = 1)
x_val = x_val.drop("SK_ID_CURR",axis = 1)
test = master_test.drop("SK_ID_CURR",axis = 1)

In [6]:
missing_pct = x_train.isnull().sum() / len(x_train) * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

cols_to_drop = missing_pct[missing_pct > 50].index
x_train = x_train.drop(columns=cols_to_drop)
x_val = x_val.drop(columns=cols_to_drop)

AMT_PAYMENT_CURRENT_std           80.305925
AMT_DRAWINGS_OTHER_CURRENT_std    80.288852
CNT_DRAWINGS_POS_CURRENT_std      80.288852
CNT_DRAWINGS_OTHER_CURRENT_std    80.288852
AMT_DRAWINGS_POS_CURRENT_std      80.288852
                                    ...    
EXT_SOURCE_2                       0.215034
AMT_GOODS_PRICE                    0.091054
AMT_ANNUITY                        0.004065
DAYS_LAST_PHONE_CHANGE             0.000406
CNT_FAM_MEMBERS                    0.000406
Length: 263, dtype: float64


In [7]:
num_cols = x_train.select_dtypes(include = [np.number]).columns
num_cols

Index(['CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
       'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
       ...
       'CNT_INSTALMENT_FUTURE_std', 'CNT_INSTALMENT_FUTURE_mean',
       'SK_DPD_min_y', 'SK_DPD_max_y', 'SK_DPD_std_y', 'SK_DPD_mean_y',
       'SK_DPD_DEF_min_y', 'SK_DPD_DEF_max_y', 'SK_DPD_DEF_std_y',
       'SK_DPD_DEF_mean_y'],
      dtype='object', length=153)

In [8]:
cat_cols = x_train.select_dtypes(include = [np.object_]).columns
cat_cols

Index(['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
       'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE',
       'EMERGENCYSTATE_MODE'],
      dtype='object')

In [9]:
for i in num_cols:
    inliers = x_train[i].quantile(0.99)
    x_train[i] = x_train[i].clip(upper = inliers)
    x_val[i] = x_val[i].clip(upper = inliers)
    test[i] = test[i].clip(upper = inliers)

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([("imputer" , SimpleImputer(strategy='median')),
                        ("scaler" , StandardScaler())])

In [11]:
from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline([("imputer" , SimpleImputer(strategy='most_frequent')),
                        ("onehot" , OneHotEncoder(handle_unknown="ignore"))])

In [12]:
from sklearn.compose import ColumnTransformer
Ct = ColumnTransformer([('nums' , num_pipeline , num_cols),
                        ('cats' , cat_pipeline , cat_cols)])
x_train_transformed = Ct.fit_transform(x_train)
x_val_transformed = Ct.transform(x_val)
test_transformed = Ct.transform(test)

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=300 , max_depth = 10 , random_state=42 , n_jobs = -1 , class_weight= "balanced")
rf_model.fit(x_train_transformed , y_train)

predictions = rf_model.predict(x_val_transformed)

probas = rf_model.predict_proba(x_val_transformed)[:, 1]

In [14]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

predictions = rf_model.predict(x_val_transformed)

acc = accuracy_score(y_val, predictions)
report = classification_report(y_val, predictions)
roc = roc_auc_score(y_val, probas) 

print("=== Classification Report ===")
print(report)
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC Score: {roc:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.95      0.77      0.85     56554
           1       0.18      0.56      0.27      4949

    accuracy                           0.76     61503
   macro avg       0.56      0.67      0.56     61503
weighted avg       0.89      0.76      0.81     61503

Accuracy: 0.7552
ROC-AUC Score: 0.7433


In [15]:
weight = (master_train["TARGET"] == 0).sum() / (master_train["TARGET"] == 1).sum()
weight

11.387150050352467

In [16]:
from xgboost import XGBClassifier

xg_model = XGBClassifier(n_estimators = 500 , max_depth = 6 , learning_rate = 0.03 , scale_pos_weight = weight )

xg_model.fit(x_train_transformed , y_train)

preds = xg_model.predict(x_val_transformed)

probapility = xg_model.predict_proba(x_val_transformed)[:,1]

In [17]:
acc = accuracy_score(y_val, preds)
report = classification_report(y_val, preds)
roc = roc_auc_score(y_val, probapility) 

print("=== Classification Report ===")
print(report)
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC Score: {roc:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.75      0.84     56554
           1       0.19      0.65      0.29      4949

    accuracy                           0.74     61503
   macro avg       0.57      0.70      0.57     61503
weighted avg       0.90      0.74      0.80     61503

Accuracy: 0.7417
ROC-AUC Score: 0.7705
